# Coleta dos 10 Ultimos jogos

In [ ]:
import pandas as pd
import numpy as np
import pyautogui
import pyperclip
import openpyxl
import re

In [ ]:
# Lidar com Tempo de espera
import time
from time import sleep

############################### SELENIUM ###############################
from selenium import webdriver # navegador

# Ações ###############################
from selenium.webdriver.common.by import By # localizar elementos
from selenium.webdriver.common.keys import Keys # comandos do teclado
from selenium.webdriver.common.action_chains import ActionChains # ações do mouse e teclado

# Exceções|Erros e Espera ###############################
from selenium.common.exceptions import NoSuchElementException # exceção para elementos não encontrados # CONTROLE DE ERROS
from selenium.webdriver.support.ui import WebDriverWait # esperar

# condições de espera ###############################
from selenium.webdriver.support.expected_conditions import (visibility_of, staleness_of, invisibility_of_element, visibility_of_element_located)
from selenium.webdriver.support import expected_conditions as EC # condições de espera (atalho)
from selenium.common.exceptions import TimeoutException # exceção para tempo limite

## ChromeDriver ###############################
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
browser = webdriver.Chrome(service=Service(ChromeDriverManager().install()))

# Abrindo site
browser.get('https://www.365scores.com/pt-br')

In [ ]:
caminho = 'C:/Users/Raphael/OneDrive/Documentos/GitHub/World_Cup_2026/Coleta de Dados/Dados em CSV/'
df = pd.read_excel('C:/Users/raphael.eugenio/Desktop/Raphael/WC26/paises.xlsx')

print(df.head())

In [ ]:
selecoes = pd.read_excel("C:/Users/raphael.eugenio/Desktop/Raphael/WC26/paises.xlsx")

In [ ]:
selecoes

In [ ]:
uefa = [
    "Bósnia e Herzegovina",
    "Suíça",
    "Escócia",
    "Turquia",
    "Alemanha",
    "Holanda",
    "Suécia",
    "Bélgica",
    "Espanha",
    "França",
    "Noruega",
    "Áustria",
    "Portugal",
    "Inglaterra",
    "Croácia",
    "Republica Tcheca"
]

conmebol = [
    "Brasil",
    "Paraguai",
    "Equador",
    "Uruguai",
    "Argentina",
    "Colômbia"
]

caf = [
    "África do Sul",
    "Marrocos",
    "Costa do Marfim",
    "Tunísia",
    "Egito",
    "Senegal",
    "Argélia",
    "RD Congo",
    "Gana",
    "Cabo Verde"
]

concacaf = [
    "México",
    "Canadá",
    "Estados Unidos",
    "Haiti",
    "Curaçao",
    "Panamá"
]

afc = [
    "Catar",
    "Austrália",
    "Japão",
    "Irã",
    "Arábia Saudita",
    "Iraque",
    "Jordânia",
    "Uzbequistão",
    "Coréia do Sul"
]

ofc = {
    "Nova Zelandia"
}

In [ ]:
for selecao in ofc:
    
    todos_dfs = []  # ← resetar AQUI, dentro do loop de seleções

    # ---- abre busca e seleciona o time ----
    browser.find_element("xpath", "/html/body/div[2]/div/div/div[1]/header/div[1]/div[2]/button").click()
    time.sleep(1)

    campo = browser.find_element("xpath", "/html/body/div[2]/div/div/div[3]/div/div[1]/div/div[2]/input")
    campo.clear()
    campo.send_keys(selecao)
    time.sleep(1)

    try:
        browser.find_element("xpath", "/html/body/div[2]/div/div/div[3]/div/div[2]/div[2]/div[2]/div[2]/div[1]/div[1]/a/div/div[2]").click()
        time.sleep(0.5)
    except:
        pyautogui.click(x=1192, y=311)
        time.sleep(0.5)

    browser.find_element("xpath", '//*[@id="sideBarModule_newCompetitor"]/div[1]/div[1]/div[2]').click()
    time.sleep(5)

    partidas_y = [310, 373, 431, 498, 564, 628, 696, 764, 834, 900]
    # partidas_y = [353, 412, 476, 543, 611, 679, 743, 809, 872, 943]
    
    for i, y in enumerate(partidas_y):

        pyautogui.click(x=685, y=y)
        time.sleep(2)

        browser.find_element("xpath", '//*[@id="navigation-tabs_game-center_stats"]/div').click()
        time.sleep(3)

        pyautogui.hotkey("ctrl", "a")
        time.sleep(0.5)
        pyautogui.hotkey("ctrl", "c")
        time.sleep(0.5)
        texto = pyperclip.paste()

        if "Top Stats" in texto:
            parte = texto.split("Top Stats")[0]
        else:
            parte = texto

        linhas = [l.strip() for l in parte.split('\n') if l.strip() != ""]

        times_encontrados = []
        for linha in reversed(linhas):
            if "National Team" in linha:
                nome_limpo = linha.replace(" National Team", "").strip()
                if nome_limpo not in times_encontrados:
                    times_encontrados.append(nome_limpo)
            if len(times_encontrados) == 2:
                break

        times_encontrados.reverse()

        time1 = times_encontrados[0] if len(times_encontrados) > 0 else None
        time2 = times_encontrados[1] if len(times_encontrados) > 1 else None
        print("Times:", time1, "x", time2)

        if time1 == selecao:
            lado_selecao = "Esquerda"
        elif time2 == selecao:
            lado_selecao = "Direita"
        else:
            lado_selecao = None
            print(f"AVISO: '{selecao}' não encontrado. Times: {time1} x {time2}")

        print(f"{selecao} está na:", lado_selecao)

        stats_texto = texto.split("Top Stats")[1]
        if "Bet365" in stats_texto:
            stats_texto = stats_texto.split("Bet365")[0]
        if "365Scores" in stats_texto:
            stats_texto = stats_texto.split("365Scores")[0]

        linhas = [l.strip() for l in stats_texto.split('\n') if l.strip() != ""]

        dados = []
        eh_numero = lambda s: bool(re.match(r'^[\d]+[\d./()%]*$', s))

        j = 0
        while j < len(linhas) - 2:
            val_a = linhas[j]
            nome  = linhas[j+1]
            val_b = linhas[j+2]
            if eh_numero(val_a) and not eh_numero(nome) and eh_numero(val_b):
                dados.append({'Metrica': nome, 'Esquerda': val_a, 'Direita': val_b})
                j += 3
            else:
                j += 1

        df = pd.DataFrame(dados)
        print(f"Stats coletadas: {len(df)} métricas")

        adversario = time2 if lado_selecao == "Esquerda" else time1

        if lado_selecao == "Esquerda":
            df_selecao = df[['Metrica', 'Esquerda']].rename(columns={'Esquerda': adversario})
        elif lado_selecao == "Direita":
            df_selecao = df[['Metrica', 'Direita']].rename(columns={'Direita': adversario})
        else:
            df_selecao = pd.DataFrame()

        df_selecao.insert(0, 'Selecao', selecao)

        if not df_selecao.empty:
            df_selecao = df_selecao.drop_duplicates(subset='Metrica', keep='first')  # ← evita duplicatas
            todos_dfs.append(df_selecao)
            print(f"Jogo {i+1} coletado: {selecao} x {adversario} ✓")
        else:
            print(f"Jogo {i+1} FALHOU — lado_selecao: {lado_selecao}")

    # ---- exporta um arquivo por seleção ----
    if todos_dfs:
        df_final = todos_dfs[0].drop_duplicates(subset='Metrica', keep='first')
        for df_next in todos_dfs[1:]:
            df_next_clean = df_next.drop_duplicates(subset='Metrica', keep='first')
            df_final = pd.merge(df_final, df_next_clean.drop(columns='Selecao'), on='Metrica', how='outer')

        df_final.to_excel(selecao + ".xlsx", index=False)
        print(f"{selecao} salvo — shape: {df_final.shape}")